In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
DB_FILE = './backend/data/bets.db' # <--- IMPORTANT: Change this to your database file name
TABLE_NAME = 'paper_bets'          # <--- IMPORTANT: Change this to your table name

# --- Load Data ---
# Connect to the SQLite database
conn = sqlite3.connect(DB_FILE)

# Create the query to select all data from the table
query = f"SELECT * FROM {TABLE_NAME}"

# Load the data into a pandas DataFrame
df = pd.read_sql_query(query, conn)

# Close the connection as we have the data in memory now
conn.close()

# Display the first few rows and data types to verify
print("Data loaded successfully. Here are the first 5 rows:")
print(df.head())
print("\nData types of the columns:")
df.info()

Data loaded successfully. Here are the first 5 rows:
   id  snapshot_id                                  match_name  \
0   1            2          Memphis Grizzlies vs Orlando Magic   
1   2            2          Memphis Grizzlies vs Orlando Magic   
2   3            4  Sacramento Kings vs Portland Trail Blazers   
3   4            4              Chicago Bulls vs Brooklyn Nets   
4   5            6              Chicago Bulls vs Brooklyn Nets   

           selection market_type  handicap  danske_odds  pinnacle_odds  \
0      Orlando Magic      Spread      -5.5         2.00           1.97   
1  Memphis Grizzlies      Spread       2.5         2.05           2.03   
2   Sacramento Kings      Spread      -4.5         2.10           2.03   
3      Chicago Bulls      Spread      -7.5         2.00           1.98   
4      Chicago Bulls      Spread      -7.5         1.98           1.97   

   ev_percent         commence_time            timestamp status  \
0        1.29  2026-01-18T17:00:00Z  2

In [2]:
# --- Time Conversion ---
# Convert string columns to datetime objects.
# For commence_time, pandas automatically detects the 'Z' as UTC.
df['commence_time'] = pd.to_datetime(df['commence_time'])

# For timestamp, we explicitly tell pandas to interpret it as UTC.
# This makes it 'tz-aware' and compatible with 'commence_time'.
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True) # <--- THIS IS THE FIX

# --- Feature Engineering ---
# 1. Calculate 'hours_to_start'
# This will now work because both columns are tz-aware in UTC.
df['hours_to_start'] = (df['commence_time'] - df['timestamp']).dt.total_seconds() / 3600

# 2. Calculate 'profit' for each bet (assuming a 1-unit stake)
conditions = [
    df['status'] == 'Won',
    df['status'] == 'Lost'
]
outcomes = [
    df['danske_odds'] - 1, # Profit if won
    -1                     # Loss if lost
]
df['profit'] = np.select(conditions, outcomes, default=0)

# --- Verification ---
# Let's check the dtypes to confirm both are tz-aware
print("Data types after timezone correction:")
print(df[['commence_time', 'timestamp']].info())

# Display the new columns to check our work
print("\nDataFrame with new 'hours_to_start' and 'profit' columns:")
print(df[['commence_time', 'timestamp', 'hours_to_start', 'status', 'danske_odds', 'profit']].head())

Data types after timezone correction:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2791 entries, 0 to 2790
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   commence_time  2791 non-null   datetime64[ns, UTC]
 1   timestamp      2791 non-null   datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](2)
memory usage: 43.7 KB
None

DataFrame with new 'hours_to_start' and 'profit' columns:
              commence_time                 timestamp  hours_to_start status  \
0 2026-01-18 17:00:00+00:00 2026-01-18 09:30:40+00:00        7.488889   Lost   
1 2026-01-18 17:00:00+00:00 2026-01-18 09:30:40+00:00        7.488889    Won   
2 2026-01-19 02:00:00+00:00 2026-01-18 13:22:07+00:00       12.631389   Lost   
3 2026-01-19 00:00:00+00:00 2026-01-18 13:22:07+00:00       10.631389    Won   
4 2026-01-19 00:00:00+00:00 2026-01-18 14:02:07+00:00        9.964722    Won   

   danske_odds  profit  
0        

In [3]:
# --- EV Threshold Binning ---

# Define the bins for ev_percent. Bins are (0, 1], (1, 2], etc.
ev_bins = [-100, 0, 1, 1.5, 2, 2.5, 3, 4, 5, 10, 100]
ev_labels = ['<0%', '0-1%', '1-1.5%', '1.5-2%', '2-2.5%', '2.5-3%', '3-4%', '4-5%', '5-10%', '>10%']

# Create a new column with the EV bin for each bet
df['ev_bin'] = pd.cut(df['ev_percent'], bins=ev_bins, labels=ev_labels, right=True)

# Group by the EV bins and calculate key metrics
ev_analysis = df.groupby('ev_bin').agg(
    num_bets=('id', 'count'),
    total_profit=('profit', 'sum')
).reset_index()

# Calculate Return on Investment (ROI) or Yield for each bin
ev_analysis['roi_percent'] = (ev_analysis['total_profit'] / ev_analysis['num_bets']) * 100

# Format for better readability
ev_analysis['total_profit'] = ev_analysis['total_profit'].round(2)
ev_analysis['roi_percent'] = ev_analysis['roi_percent'].round(2)

print("\n--- EV Threshold Analysis ---")
print(ev_analysis)


--- EV Threshold Analysis ---
   ev_bin  num_bets  total_profit  roi_percent
0     <0%         0          0.00          NaN
1    0-1%       986        -53.15        -5.39
2  1-1.5%       485        -45.00        -9.28
3  1.5-2%       311        -12.82        -4.12
4  2-2.5%       231        -10.37        -4.49
5  2.5-3%       113         -4.41        -3.90
6    3-4%       387        -19.82        -5.12
7    4-5%       130         -0.50        -0.38
8   5-10%       143        -17.95       -12.55
9    >10%         5          1.85        37.00


/tmp/ipykernel_65522/651578423.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ev_analysis = df.groupby('ev_bin').agg(


In [4]:
# --- Time-to-Start Binning ---

# Define the bins for hours_to_start. Bins are (0, 1], (1, 3], etc.
time_bins = [0, 1, 3, 6, 12, 24, 48, 72, 10000] # 10000 acts as infinity
time_labels = ['0-1h', '1-3h', '3-6h', '6-12h', '12-24h', '24-48h', '48-72h', '>72h']

# Create a new column with the Time bin for each bet
df['time_bin'] = pd.cut(df['hours_to_start'], bins=time_bins, labels=time_labels, right=False)

# Group by the time bins and calculate key metrics
time_analysis = df.groupby('time_bin').agg(
    num_bets=('id', 'count'),
    total_profit=('profit', 'sum')
).reset_index()

# Calculate ROI for each bin
time_analysis['roi_percent'] = (time_analysis['total_profit'] / time_analysis['num_bets']) * 100

# Format for better readability
time_analysis['total_profit'] = time_analysis['total_profit'].round(2)
time_analysis['roi_percent'] = time_analysis['roi_percent'].round(2)

print("\n--- Time-to-Start Analysis ---")
print(time_analysis)


--- Time-to-Start Analysis ---
  time_bin  num_bets  total_profit  roi_percent
0     0-1h       138          0.23         0.17
1     1-3h       292         -0.43        -0.15
2     3-6h       574        -42.40        -7.39
3    6-12h      1009       -114.04       -11.30
4   12-24h       778         -5.53        -0.71
5   24-48h         0          0.00          NaN
6   48-72h         0          0.00          NaN
7     >72h         0          0.00          NaN


/tmp/ipykernel_65522/3770745659.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  time_analysis = df.groupby('time_bin').agg(


In [5]:
from IPython.display import display, Markdown

# Configure pandas to display floats with 2 decimal places
pd.options.display.float_format = '{:,.2f}'.format

# Helper function to print a header before a table
def display_header(text):
    display(Markdown(f"### {text}"))

In [6]:
import matplotlib.pyplot as plt

# --- 1. Create the ROI Pivot Table ---
roi_pivot = df.pivot_table(
    index='ev_bin',
    columns='time_bin',
    values='profit',
    aggfunc=lambda x: (x.sum() / x.count()) * 100, # Calculates ROI %
    observed=True
)

# --- 2. Create the Sample Size (Count) Pivot Table ---
count_pivot = df.pivot_table(
    index='ev_bin',
    columns='time_bin',
    values='profit',
    aggfunc='count',
    observed=True
).fillna(0).astype(int)

# --- Display the Tables ---
display_header("ROI % Heatmap (EV vs. Time-to-Start)")
display(Markdown("Green cells indicate profitable combinations. **Always cross-reference with the sample size table below!**"))

# Use a styler to apply the color gradient (green for high, red for low)
display(
    roi_pivot.style
    .background_gradient(cmap='RdYlGn', axis=None) 
    .format("{:.2f}%", na_rep="-")
)

display_header("Number of Bets per Cell (Sample Size)")
display(Markdown("Use this table to validate the heatmap above. Ignore cells with very few bets (e.g., < 20)."))
display(count_pivot.style.background_gradient(cmap='Blues', axis=None))

### ROI % Heatmap (EV vs. Time-to-Start)

Green cells indicate profitable combinations. **Always cross-reference with the sample size table below!**

time_bin,0-1h,1-3h,3-6h,6-12h,12-24h
ev_bin,,,,,
0-1%,-9.03%,-1.01%,-12.08%,-8.87%,2.89%
1-1.5%,4.55%,3.59%,-3.93%,-13.76%,-16.82%
1.5-2%,0.27%,14.39%,-3.42%,-10.43%,-7.14%
2-2.5%,11.36%,-30.96%,1.53%,5.85%,-15.68%
2.5-3%,-15.00%,-12.27%,-15.31%,-7.34%,25.18%
3-4%,3.33%,15.72%,-3.11%,-20.84%,4.28%
4-5%,-17.50%,2.86%,-5.54%,-21.35%,28.21%
5-10%,18.75%,-33.64%,-11.45%,-23.44%,-2.11%
>10%,125.00%,-,-100.00%,17.50%,125.00%


### Number of Bets per Cell (Sample Size)

Use this table to validate the heatmap above. Ignore cells with very few bets (e.g., < 20).

time_bin,0-1h,1-3h,3-6h,6-12h,12-24h
ev_bin,,,,,
0-1%,36,100,200,370,280
1-1.5%,22,56,114,188,105
1.5-2%,11,46,48,95,111
2-2.5%,11,25,45,88,62
2.5-3%,6,15,26,44,22
3-4%,33,32,72,137,113
4-5%,10,7,37,37,39
5-10%,8,11,31,48,45
>10%,1,0,1,2,1


In [42]:
# Define a function to filter profitable bets
def filter_profitable_bets(df, roi_threshold=0, sample_size_threshold=20):
    # Initialize an empty DataFrame to store profitable bets
    profitable_bets = pd.DataFrame(columns=df.columns, index=df.index)

    # Iterate through the DataFrame to check each cell
    for ev_bin in df.index:
        for time_bin in df.columns:
            roi = df.loc[ev_bin, time_bin]
            sample_size = count_pivot.loc[ev_bin, time_bin]
            if roi > roi_threshold and sample_size >= sample_size_threshold:
                profitable_bets.loc[ev_bin, time_bin] = roi

    return profitable_bets

# Apply the function to your data
profitable_bets = filter_profitable_bets(roi_pivot)

# Display the profitable bets
print("Profitable Bets:")
print(profitable_bets)

# Create a strategy based on the profitable bets
strategy = {
    'ev_bin': profitable_bets.index.tolist(),
    'time_bin': profitable_bets.columns.tolist(),
    'roi_percent': profitable_bets.to_dict()
}

Profitable Bets:
time_bin 0-1h  1-3h 3-6h 6-12h 12-24h
ev_bin                               
0-1%      NaN   NaN  NaN   NaN   2.89
1-1.5%   4.55  3.59  NaN   NaN    NaN
1.5-2%    NaN 14.39  NaN   NaN    NaN
2-2.5%    NaN   NaN 1.53  5.85    NaN
2.5-3%    NaN   NaN  NaN   NaN  25.18
3-4%     3.33 15.72  NaN   NaN   4.28
4-5%      NaN   NaN  NaN   NaN  28.21
5-10%     NaN   NaN  NaN   NaN    NaN
>10%      NaN   NaN  NaN   NaN    NaN


In [59]:
# Filter the DataFrame for the specified time and EV bins
filtered_df = df[(df['hours_to_start'] >= 11) &
                  (df['ev_percent'] > 2.5) &
                  (df['status'] != 'Pending')]

# Remove later bets on the same game
filtered_df = filtered_df.drop_duplicates(subset=['match_name', 'commence_time'], keep='first')

# Calculate the sum of ROI, excluding NaN values
total_roi = filtered_df['profit'].sum() / filtered_df['profit'].count() * 100

# Print the result
print(f"Total ROI for 12-24h window and 2.5% and above threshold: {total_roi:.2f}%")
print(f"Number of bets: {len(filtered_df)}")


Total ROI for 12-24h window and 2.5% and above threshold: 10.56%
Number of bets: 176


In [47]:
filtered_df.loc[:, ['match_name', 'selection', 'market_type', 'handicap', 'commence_time', 'status', 'result_score']].to_csv('QA.csv')

In [39]:
import sklearn

# Check for data anomalies
print("Data Anomalies:")
print(filtered_df.describe())

# Time period analysis
filtered_df['month'] = filtered_df['commence_time'].dt.to_period('M')
monthly_roi = filtered_df.groupby('month').agg(
    total_profit=('profit', 'sum'),
    num_bets=('id', 'count')
).reset_index()
monthly_roi['roi_percent'] = (monthly_roi['total_profit'] / monthly_roi['num_bets']) * 100
print("\nMonthly ROI:")
print(monthly_roi)

# Sample size analysis
print("\nSample Size Analysis:")
print(filtered_df['profit'].describe())

# Cross-validation
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(filtered_df, test_size=0.2, random_state=42)
train_roi = train_df['profit'].sum() / train_df['profit'].count() * 100
test_roi = test_df['profit'].sum() / test_df['profit'].count() * 100
print(f"\nTrain ROI: {train_roi:.2f}%")
print(f"Test ROI: {test_roi:.2f}%")

# Statistical significance
from scipy import stats
t_stat, p_value = stats.ttest_1samp(filtered_df['profit'], 0)
print(f"\nStatistical Significance: t-stat = {t_stat:.2f}, p-value = {p_value:.4f}")

# Compare with baseline
baseline_roi = df['profit'].sum() / df['profit'].count() * 100
print(f"\nBaseline ROI: {baseline_roi:.2f}%")

Data Anomalies:
            id  snapshot_id  handicap  danske_odds  pinnacle_odds  ev_percent  \
count   155.00       155.00    155.00       155.00         155.00      155.00   
mean  1,220.70       377.80     -3.38         2.07           1.99        4.28   
std     719.65       235.39      7.66         0.05           0.04        1.67   
min       3.00         4.00    -19.50         1.98           1.91        2.55   
25%     616.50       186.50     -9.50         2.05           1.96        3.11   
50%   1,171.00       339.00     -5.50         2.05           1.98        3.82   
75%   1,789.50       542.00      3.50         2.10           2.01        4.67   
max   2,626.00       907.00     16.50         2.25           2.08       13.95   

       closing_odds  hours_to_start  profit  
count        127.00          155.00  155.00  
mean           2.00           15.40    0.12  
std            0.04            2.31    1.04  
min            1.92           12.19   -1.00  
25%            1.97     

/tmp/ipykernel_65522/3812856981.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  filtered_df['month'] = filtered_df['commence_time'].dt.to_period('M')


In [48]:
"""
NBA Bet Data Validation Script
===============================
Validates:
  1. Spread bet outcomes (Won/Lost) are correctly calculated from scores.
  2. Recorded scores match official NBA data (via nba_api).

Requirements:
  pip install pandas nba_api
"""

import pandas as pd
import re
import time
from zoneinfo import ZoneInfo
from nba_api.stats.endpoints import leaguegamefinder

# ── Configuration ────────────────────────────────────────────────────────────
CSV_PATH = "QA.csv"
NBA_TIMEZONE = ZoneInfo("America/New_York")
API_DELAY_SECONDS = 1  # delay between API calls to avoid rate-limiting


# ── Parsing helpers ──────────────────────────────────────────────────────────

SCORE_PATTERN = re.compile(r"^(.+?)\s+(\d+)\s*-\s*(\d+)\s+(.+)$")

# nba_api sometimes uses shortened names; map your CSV names to API names
# Add entries here only if you encounter "not found" issues
TEAM_NAME_MAP = {
    # "csv_name": "api_name",
    # e.g. "LA Clippers": "Los Angeles Clippers",
}


def normalize_team_name(name: str) -> str:
    """Apply team name corrections if needed."""
    return TEAM_NAME_MAP.get(name, name)


def parse_result_score(result_score: str) -> dict | None:
    """
    Parse a string like:
      'Sacramento Kings 110 - 117 Portland Trail Blazers'
    into its components.
    """
    m = SCORE_PATTERN.match(result_score.strip())
    if not m:
        return None
    return {
        "home_team": m.group(1).strip(),
        "home_score": int(m.group(2)),
        "away_score": int(m.group(3)),
        "away_team": m.group(4).strip(),
    }


def get_selection_scores(parsed: dict, selection: str) -> tuple[int, int] | tuple[None, None]:
    """Return (selection_score, opponent_score)."""
    if selection == parsed["home_team"]:
        return parsed["home_score"], parsed["away_score"]
    elif selection == parsed["away_team"]:
        return parsed["away_score"], parsed["home_score"]
    return None, None


def expected_spread_result(sel_score: int, opp_score: int, handicap: float) -> str:
    margin = sel_score - opp_score + handicap
    if margin > 0:
        return "Won"
    elif margin < 0:
        return "Lost"
    return "Push"


def commence_to_nba_date(ts) -> str | None:
    """
    Convert a UTC commence_time to the NBA game-date in Eastern Time.
    Returns 'MM/DD/YYYY' string (format expected by nba_api).
    """
    if pd.isna(ts):
        return None
    et = ts.astimezone(NBA_TIMEZONE)
    return et.strftime("%m/%d/%Y")


# ── NBA API helpers ──────────────────────────────────────────────────────────

def fetch_games_for_date(date_str: str) -> pd.DataFrame:
    """
    Query LeagueGameFinder for a single date.
    Returns a DataFrame with columns like TEAM_NAME, GAME_ID, PTS, etc.
    """
    try:
        result = leaguegamefinder.LeagueGameFinder(
            date_from_nullable=date_str,
            date_to_nullable=date_str,
            league_id_nullable="00",  # 00 = NBA
        )
        return result.get_data_frames()[0]
    except Exception as e:
        print(f"    [ERROR] API request failed for {date_str}: {e}")
        return pd.DataFrame()


def build_api_score_lookup(api_df: pd.DataFrame) -> dict:
    """
    Build a lookup:  (team_name, game_id) -> points scored
    so we can quickly find any team's score in any game.
    """
    lookup = {}
    for _, row in api_df.iterrows():
        lookup[(row["TEAM_NAME"], row["GAME_ID"])] = int(row["PTS"])
    return lookup


def find_game_scores(
    api_df: pd.DataFrame,
    lookup: dict,
    home_team: str,
    away_team: str,
) -> tuple[int, int] | tuple[None, None]:
    """
    Find official (home_score, away_score) for a specific matchup in the API data.
    Matches on GAME_ID shared by both teams.
    """
    home_team = normalize_team_name(home_team)
    away_team = normalize_team_name(away_team)

    home_rows = api_df[api_df["TEAM_NAME"] == home_team]
    away_rows = api_df[api_df["TEAM_NAME"] == away_team]

    common_ids = set(home_rows["GAME_ID"]) & set(away_rows["GAME_ID"])
    if not common_ids:
        return None, None

    gid = common_ids.pop()
    return lookup.get((home_team, gid)), lookup.get((away_team, gid))


# ── Main validation ──────────────────────────────────────────────────────────

def main():
    # ── Load CSV ─────────────────────────────────────────────────────────────
    print("=" * 70)
    print("STEP 1 : Load CSV")
    print("=" * 70)

    df = pd.read_csv(CSV_PATH, index_col=0)
    df["commence_time"] = pd.to_datetime(df["commence_time"], utc=True)
    print(f"  Rows loaded        : {len(df)}")
    print(f"  Date range         : {df['commence_time'].min().date()} → "
          f"{df['commence_time'].max().date()}")

    # ── Parse scores ─────────────────────────────────────────────────────────
    df["_parsed"] = df["result_score"].apply(parse_result_score)
    parse_fail_mask = df["_parsed"].isna()
    if parse_fail_mask.any():
        print(f"\n  [WARN] Could not parse result_score for rows: "
              f"{list(df[parse_fail_mask].index)}")
    print()

    # ── Offline spread-logic check ───────────────────────────────────────────
    print("=" * 70)
    print("STEP 2 : Validate spread logic (offline)")
    print("=" * 70)

    spread_errors = []
    for idx, row in df[~parse_fail_mask].iterrows():
        p = row["_parsed"]
        sel_sc, opp_sc = get_selection_scores(p, row["selection"])
        if sel_sc is None:
            spread_errors.append((idx, "selection not found in result_score"))
            continue
        exp = expected_spread_result(sel_sc, opp_sc, row["handicap"])
        if exp != row["status"]:
            spread_errors.append(
                (idx, f"{row['match_name']} | sel={row['selection']} "
                      f"hcap={row['handicap']} score={sel_sc}-{opp_sc} "
                      f"expected={exp} got={row['status']}")
            )

    if spread_errors:
        print(f"\n  [FAIL] {len(spread_errors)} spread-logic errors:")
        for i, msg in spread_errors:
            print(f"    Row {i}: {msg}")
    else:
        ok = (~parse_fail_mask).sum()
        print(f"\n  [PASS] All {ok} spread outcomes are correct.")
    print()

    # ── Online score verification ────────────────────────────────────────────
    print("=" * 70)
    print("STEP 3 : Verify scores via NBA API")
    print("=" * 70)

    df["_game_date"] = df["commence_time"].apply(commence_to_nba_date)
    unique_dates = sorted(df["_game_date"].dropna().unique())
    print(f"  Unique game dates to query: {len(unique_dates)}\n")

    # Fetch & cache API data per date
    api_cache: dict[str, tuple[pd.DataFrame, dict]] = {}
    for d in unique_dates:
        print(f"  Fetching {d} …", end=" ")
        api_df = fetch_games_for_date(d)
        if api_df.empty:
            print("no data")
        else:
            print(f"{len(api_df)} team-game rows")
            api_cache[d] = (api_df, build_api_score_lookup(api_df))
        time.sleep(API_DELAY_SECONDS)

    print()

    verified = 0
    mismatches = []
    not_found = []

    for idx, row in df[~parse_fail_mask].iterrows():
        p = row["_parsed"]
        gd = row["_game_date"]
        if gd not in api_cache:
            not_found.append(idx)
            continue

        api_df, lookup = api_cache[gd]
        api_home, api_away = find_game_scores(
            api_df, lookup, p["home_team"], p["away_team"]
        )

        if api_home is None:
            not_found.append(idx)
            continue

        if api_home == p["home_score"] and api_away == p["away_score"]:
            verified += 1
        else:
            mismatches.append({
                "index": idx,
                "match": row["match_name"],
                "csv": f"{p['home_team']} {p['home_score']} - "
                       f"{p['away_score']} {p['away_team']}",
                "api": f"{p['home_team']} {api_home} - "
                       f"{api_away} {p['away_team']}",
            })

    # ── Summary ──────────────────────────────────────────────────────────────
    print("=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)

    total = len(df)
    parse_fail = int(parse_fail_mask.sum())

    print(f"\n  Total rows           : {total}")
    print(f"  Parse failures       : {parse_fail}")
    print()
    print("  Spread logic")
    print(f"    ✅ Correct          : {total - parse_fail - len(spread_errors)}")
    print(f"    ❌ Wrong            : {len(spread_errors)}")
    print()
    print("  Score verification (API)")
    print(f"    ✅ Verified         : {verified}")
    print(f"    ❌ Mismatched       : {len(mismatches)}")
    print(f"    ⚠️  Not found       : {len(not_found)}")

    if mismatches:
        print("\n  Score mismatches:")
        for mm in mismatches:
            print(f"    Row {mm['index']}: {mm['match']}")
            print(f"      CSV : {mm['csv']}")
            print(f"      API : {mm['api']}")

    if not_found:
        preview = not_found[:20]
        print(f"\n  Could not match to API: {preview}"
              f"{'…' if len(not_found) > 20 else ''}")

    print()
    if not spread_errors and not mismatches:
        print("  🎉  All checks passed — your pipeline looks correct!")
    else:
        print("  ⚠️   Issues detected — review details above.")


if __name__ == "__main__":
    main()

STEP 1 : Load CSV
  Rows loaded        : 155
  Date range         : 2026-01-19 → 2026-04-11

STEP 2 : Validate spread logic (offline)

  [PASS] All 155 spread outcomes are correct.

STEP 3 : Verify scores via NBA API
  Unique game dates to query: 64

  Fetching 01/18/2026 … 12 team-game rows
  Fetching 01/19/2026 … 18 team-game rows
  Fetching 01/20/2026 … 14 team-game rows
  Fetching 01/21/2026 … 14 team-game rows
  Fetching 01/22/2026 … 16 team-game rows
  Fetching 01/23/2026 … 16 team-game rows
  Fetching 01/24/2026 … 12 team-game rows
  Fetching 01/25/2026 … 12 team-game rows
  Fetching 01/27/2026 … 14 team-game rows
  Fetching 01/28/2026 … 18 team-game rows
  Fetching 01/29/2026 … 16 team-game rows
  Fetching 01/30/2026 … 18 team-game rows
  Fetching 01/31/2026 … 12 team-game rows
  Fetching 02/01/2026 … 20 team-game rows
  Fetching 02/02/2026 … 8 team-game rows
  Fetching 02/03/2026 … 20 team-game rows
  Fetching 02/04/2026 … 14 team-game rows
  Fetching 02/05/2026 … 16 team-game